# Zonificación de la Susceptibilidad a Inundación en la Provincia de Santa Elena - Ecuador
### Modelo Hidrológico Espacial Replicable: Extracción, Simulación SCS-CN Multiescenario, Figuras Científicas (300 DPI) y Visor WebGIS Standalone
**Elaborado por:** Bruce Tumbaco Vega  
**Línea de Investigación:** Hidrología Espacial, Teledetección y Modelado de Crecidas  

> **Nota de Replicabilidad para Evaluadores:** Este cuaderno reproduce de manera autónoma y celda a celda todas las tablas científicas (Tablas 1, 2, 3 y 4), las cuatro figuras de alta resolución a 300 DPI en inglés (Figuras 1, 2, 3 y 4) y el visor interactivo cartográfico WebGIS standalone (`visor_inundaciones_santa_elena.html`), empleando directamente los shapefiles y rásters oficiales de Santa Elena incluidos en el paquete `santa_elena_datos.zip`.


## Paso 1: Instalación de dependencias geoespaciales y configuración editorial


In [ ]:
# Instalar dependencias geoespaciales avanzadas para análisis ráster y vectorial
!pip install -q geopandas rasterio shapely matplotlib scipy pillow

# Importar librerías de cálculo científico, manejo geoespacial y visualización
import os                                      # Gestión de rutas y sistema de archivos
import json                                    # Serialización y estructuración de datos JSON
import zipfile                                 # Extracción y compresión de archivos ZIP
import base64                                  # Codificación de rásters binarios a base64 para WebGIS
from io import BytesIO                         # Manejo de búferes de memoria en RAM
import numpy as np                             # Operaciones matriciales y cálculo numérico
import pandas as pd                            # Manipulación y formateo de tablas científicas
import geopandas as gpd                        # Lectura y procesamiento de vectores geoespaciales
from shapely.geometry import box, Point        # Creación de geometrías de polígonos y puntos
import rasterio                                # Lectura e indexación de mallas ráster GeoTIFF
from rasterio.features import geometry_mask    # Generación de máscaras espaciales de polígonos
import matplotlib.pyplot as plt                # Renderizado de figuras científicas de alta calidad
import matplotlib.colors as mcolors            # Definición y normalización de rampas de color
from matplotlib.patches import Patch           # Construcción de parches para leyendas
from matplotlib.lines import Line2D            # Construcción de líneas para leyendas

# Configurar parámetros tipográficos y de resolución para figuras de calidad editorial (300 DPI)
plt.rcParams["font.family"] = "sans-serif"     # Tipografía principal sin serifa
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial"] # Fuentes compatibles con Linux y Windows
plt.rcParams["axes.edgecolor"] = "#334155"     # Color técnico perimetral de ejes
plt.rcParams["axes.linewidth"] = 0.8           # Grosor de línea de marcos
plt.rcParams["figure.dpi"] = 300               # Resolución por defecto a 300 DPI
print("Entorno geoespacial configurado exitosamente a 300 DPI.")


## Paso 2: Carga y descompresión del paquete de datos oficiales de Santa Elena


In [ ]:
# Definir nombre del archivo ZIP con los datos geoespaciales oficiales de Santa Elena
zip_file = "santa_elena_datos.zip"              # Archivo ZIP que acompaña al cuaderno
data_dir = "datos_santa_elena"                 # Carpeta destino para extraer los datos

# Soporte flexible para rutas relativas tanto en Colab como en local
if not os.path.exists(zip_file) and os.path.exists(os.path.join("Inundaciones_Santa_Elena", "santa_elena_datos.zip")):
    zip_file = os.path.join("Inundaciones_Santa_Elena", "santa_elena_datos.zip")
if not os.path.exists(data_dir) and os.path.exists("datos_gee"):
    data_dir = "datos_gee"

# Si no existe la carpeta ni el zip, solicitar subida interactiva en Google Colab
if not os.path.exists(zip_file) and not os.path.exists(data_dir):
    print(f"Por favor, suba el archivo '{zip_file}' que acompaña al proyecto:")
    try:
        from google.colab import files         # Importar módulo de subida de Colab
        uploaded = files.upload()              # Abrir cuadro de diálogo de subida
    except Exception as e:
        print(f"Asegúrese de colocar '{zip_file}' en el directorio de trabajo.")

# Descomprimir los shapefiles y rásters oficiales si existe el ZIP
if os.path.exists(zip_file):
    with zipfile.ZipFile(zip_file, "r") as z:  # Abrir archivo ZIP
        z.extractall(data_dir)                 # Extraer todo el contenido
    print(f"Paquete descomprimido exitosamente en '{data_dir}/': {len(os.listdir(data_dir))} archivos.")
else:
    print(f"Usando datos existentes en la carpeta '{data_dir}/'.")


## Paso 3: Carga de límites vectoriales oficiales (Shapefile), rásters GEE y Tabla 1 del Artículo


In [ ]:
# Cargar geometrías vectoriales oficiales de la provincia de Santa Elena
shp_path = os.path.join(data_dir, "santa_elena_utm17s.shp") # Ruta al Shapefile oficial
se_shp = gpd.read_file(shp_path) if os.path.exists(shp_path) else None # Cargar Shapefile

# Cargar capas GeoJSON de límites provinciales, cantonales y red hidrográfica
se_wgs84 = gpd.read_file(os.path.join(data_dir, "santa_elena_wgs84.geojson")) # Límite provincial WGS84
cantones = gpd.read_file(os.path.join(data_dir, "cantones_santa_elena.geojson")) # Límites cantonales INEC
rivers = gpd.clip(gpd.read_file(os.path.join(data_dir, "rios_hydrosheds.geojson")), se_wgs84) # Red fluvial

# Leer las 4 pilas de mallas ráster multibanda de 90 m procesadas en Google Earth Engine
with rasterio.open(os.path.join(data_dir, "01_topo_hydro_90m.tif")) as src1:
    transform = src1.transform                 # Transformación afín espacial
    bounds = src1.bounds                       # Bounding box geográfica
    elev = src1.read(1)                        # Banda 1: Elevación SRTM (m s.n.m.)
    slope = src1.read(2)                       # Banda 2: Pendiente topográfica (grados)
    hand = src1.read(3)                        # Banda 3: Altura sobre drenaje HAND (m)
    upa = src1.read(4)                         # Banda 4: Área de drenaje acumulada (km²)
    river_width = src1.read(5)                 # Banda 5: Ancho estimado de cauce (m)
    twi = src1.read(6)                         # Banda 6: Índice Topográfico de Humedad TWI

with rasterio.open(os.path.join(data_dir, "02_lulc_clim_90m.tif")) as src2:
    lulc = src2.read(1)                        # Banda 1: Cobertura de suelo ESA WorldCover 10 m
    jrc_occ = src2.read(2)                     # Banda 2: Frecuencia de agua superficial JRC (%)

with rasterio.open(os.path.join(data_dir, "03_sentinel2_indices_90m.tif")) as src3:
    ndvi_wet = src3.read(1)                    # Banda 1: NDVI época lluviosa
    ndwi_wet = src3.read(2)                    # Banda 2: NDWI época lluviosa
    mndwi_wet = src3.read(3)                   # Banda 3: MNDWI época lluviosa
    ndbi_wet = src3.read(4)                    # Banda 4: NDBI época lluviosa
    mndwi_dry = src3.read(6)                   # Banda 6: MNDWI época seca

with rasterio.open(os.path.join(data_dir, "04_sentinel1_sar_90m.tif")) as src4:
    s1_wet = src4.read(1)                      # Banda 1: SAR VV época húmeda (dB)
    s1_dry = src4.read(2)                      # Banda 2: SAR VV época seca (dB)

# Generar máscara booleana exacta de tierra firme para la provincia
prov_mask = ~geometry_mask(se_wgs84.geometry, out_shape=elev.shape, transform=transform, invert=False)
valid_pixels = int(np.sum(prov_mask))          # Número total de celdas válidas en la provincia
pixel_area_ha = 369695.9 / valid_pixels        # Área real por celda (ha)
pixel_area_m2 = pixel_area_ha * 10000.0        # Área real por celda en m²

# Función de limpieza y filtrado de valores válidos dentro de la máscara provincial
def clean_band(arr, default_val=0.0):
    out = np.where(np.isfinite(arr), arr, default_val)
    return np.where(prov_mask, out, np.nan)

elev = clean_band(elev, 0.0)
slope = clean_band(slope, 0.1)
hand = clean_band(np.maximum(hand, 0.0), 0.0)
upa = clean_band(np.maximum(upa, 0.01), 0.01)
twi = clean_band(twi, 8.0)
lulc = clean_band(lulc, 30.0)
jrc_occ = clean_band(jrc_occ, 0.0)
mndwi_wet = clean_band(mndwi_wet, -0.25)
s1_wet = clean_band(s1_wet, -12.0)

# TABLA 1 DEL ARTÍCULO CIENTÍFICO: Fuentes y colecciones satelitales en Google Earth Engine
tabla_1_data = [
    {"Fuente de Datos / Sensor": "Sentinel-1 SAR (C-band)", "Colección en GEE / ID": "COPERNICUS/S1_GRD", "Resolución Espacial": "10 m", "Resolución Temporal": "6 a 12 días", "Variables Derivadas": "Retrodispersión VV (dB), compuestos mín/máx, atenuación especular"},
    {"Fuente de Datos / Sensor": "Sentinel-2 MSI (L2A)", "Colección en GEE / ID": "COPERNICUS/S2_SR_HARMONIZED", "Resolución Espacial": "10 a 20 m", "Resolución Temporal": "5 días", "Variables Derivadas": "Reflectancia BOA, MNDWI, NDWI, NDVI, NDBI"},
    {"Fuente de Datos / Sensor": "MERIT Hydro DEM", "Colección en GEE / ID": "MERIT/Hydro/v1_0_1", "Resolución Espacial": "90 m (~3 arc-sec)", "Resolución Temporal": "Estático", "Variables Derivadas": "Elevación ajustada, HAND, TWI, red fluvial, acumulación UPA"},
    {"Fuente de Datos / Sensor": "SRTM GL1 DEM", "Colección en GEE / ID": "USGS/SRTMGL1_003", "Resolución Espacial": "30 m (1 arc-sec)", "Resolución Temporal": "Estático", "Variables Derivadas": "Pendiente topográfica (° y %), relieve sombreado"},
    {"Fuente de Datos / Sensor": "JRC Global Surface Water", "Colección en GEE / ID": "JRC/GSW1_4/GlobalSurfaceWater", "Resolución Espacial": "30 m", "Resolución Temporal": "38 años (1984 a 2021)", "Variables Derivadas": "Frecuencia de ocurrencia de agua superficial (%), recurrencia"},
    {"Fuente de Datos / Sensor": "ESA WorldCover 2021", "Colección en GEE / ID": "ESA/WorldCover/v200", "Resolución Espacial": "10 m", "Resolución Temporal": "Anual (2021)", "Variables Derivadas": "Cobertura/uso de suelo (11 clases), impermeabilidad urbana"},
    {"Fuente de Datos / Sensor": "ERA5-Land / GPM IMERG", "Colección en GEE / ID": "ECMWF/ERA5_LAND/DAILY_RAW", "Resolución Espacial": "9 a 11 km", "Resolución Temporal": "Horaria / Diaria", "Variables Derivadas": "Precipitación histórica extrema, eventos de tormenta 120 mm"}
]
df_tabla_1 = pd.DataFrame(tabla_1_data)
df_tabla_1.to_csv("tabla_1_datos_gee.csv", index=False)
print("=== TABLA 1 DEL ARTÍCULO: CONJUNTOS DE DATOS GEOESPACIALES Y COLECCIONES GEE ===")
print(df_tabla_1.to_string(index=False))


## Paso 4: Modelado del Índice FSI (AHP) y Tabla 2 del Artículo (Desglose Cantonal)


In [ ]:
# Construir mapa de Curva Número (SCS-CN) según cobertura ESA WorldCover y topografía
cn_map = np.full_like(elev, 82.0)
cn_lookup = {10: 73.0, 20: 79.0, 30: 82.0, 40: 85.0, 50: 93.0, 60: 89.0, 80: 98.0, 90: 94.0, 95: 91.0}
for code, val in cn_lookup.items():
    cn_map = np.where((lulc == code) & prov_mask, val, cn_map)
cn_map = np.where((hand < 3.0) & (slope < 2.0) & (cn_map < 94.0), np.minimum(95.0, cn_map + 4.0), cn_map)
cn_map = np.where(prov_mask, cn_map, np.nan)
S_mm = 25400.0 / cn_map - 254.0                # Retención máxima potencial del suelo
Ia_mm = 0.2 * S_mm                             # Abstracciones iniciales

# Normalizar factores biofísicos al rango [0, 1]
def norm01(arr, vmin, vmax, invert=False):
    n = np.clip((arr - vmin) / (vmax - vmin + 1e-9), 0.0, 1.0)
    return np.where(prov_mask, 1.0 - n if invert else n, np.nan)

f_hand = norm01(hand, 0.0, 15.0, invert=True)
f_twi = norm01(twi, 7.0, 18.0, invert=False)
f_slope = norm01(slope, 0.0, 12.0, invert=True)
f_elev = norm01(elev, 0.0, 80.0, invert=True)
f_mndwi = norm01(mndwi_wet, -0.40, 0.15, invert=False)
f_sar = norm01(s1_wet, -20.0, -8.0, invert=True)
f_cn = norm01(cn_map, 70.0, 95.0, invert=False)

# Ponderación AHP (Proceso de Jerarquía Analítica, CR = 0.038)
fsi = (0.28 * f_hand + 0.20 * f_twi + 0.14 * f_slope + 0.10 * f_elev + 0.12 * f_mndwi + 0.08 * f_sar + 0.08 * f_cn)
fsi = np.where(prov_mask, fsi, np.nan)

# Clasificación en 5 categorías estándar
fsi_class = np.zeros_like(fsi)
fsi_class = np.where((fsi < 0.20) & prov_mask, 1, fsi_class)
fsi_class = np.where((fsi >= 0.20) & (fsi < 0.35) & prov_mask, 2, fsi_class)
fsi_class = np.where((fsi >= 0.35) & (fsi < 0.50) & prov_mask, 3, fsi_class)
fsi_class = np.where((fsi >= 0.50) & (fsi < 0.65) & prov_mask, 4, fsi_class)
fsi_class = np.where((fsi >= 0.65) & prov_mask, 5, fsi_class)

# TABLA 2 DEL ARTÍCULO CIENTÍFICO: Distribución de FSI por Cantón y Total Provincial
classes_def = [
    ("Muy Baja (< 0.20)", 0.00, 0.20),
    ("Baja (0.20 a 0.35)", 0.20, 0.35),
    ("Moderada (0.35 a 0.50)", 0.35, 0.50),
    ("Alta (0.50 a 0.65)", 0.50, 0.65),
    ("Muy Alta / Crítica (≥ 0.65)", 0.65, 1.01)
]

canton_names = ["Santa Elena", "Salinas", "La Libertad"]
canton_masks = {}
for cn in canton_names:
    geom = cantones[cantones["ADM2_NAME"] == cn].geometry.values[0]
    m = (~geometry_mask([geom], out_shape=elev.shape, transform=transform, invert=False)) & prov_mask
    canton_masks[cn] = m

tot_prov_ha = np.sum(prov_mask) * pixel_area_ha
tabla_2_rows = []

for cname, low, high in classes_def:
    m_prov = (fsi >= low) & (fsi < high) & prov_mask
    ha_prov = np.sum(m_prov) * pixel_area_ha
    pct_prov = (ha_prov / tot_prov_ha) * 100.0
    r = {
        "Nivel de Susceptibilidad": cname,
        "Provincial (ha / %)": f"{ha_prov:,.1f} ha ({pct_prov:.1f}%)"
    }
    for cn in canton_names:
        m_c = canton_masks[cn]
        tot_c = np.sum(m_c) * pixel_area_ha
        ha_c = np.sum(m_prov & m_c) * pixel_area_ha
        pct_c = (ha_c / tot_c) * 100.0 if tot_c > 0 else 0.0
        r[f"{cn} (ha / %)"] = f"{ha_c:,.1f} ha ({pct_c:.1f}%)"
    tabla_2_rows.append(r)

# Fila de Totales
r_tot = {
    "Nivel de Susceptibilidad": "Total Cantonal / Provincial",
    "Provincial (ha / %)": f"{tot_prov_ha:,.1f} ha (100.0%)"
}
for cn in canton_names:
    tot_c = np.sum(canton_masks[cn]) * pixel_area_ha
    r_tot[f"{cn} (ha / %)"] = f"{tot_c:,.1f} ha (100.0%)"
tabla_2_rows.append(r_tot)

df_tabla_2 = pd.DataFrame(tabla_2_rows)
df_tabla_2.to_csv("tabla_2_cantones_fsi.csv", index=False)
print("=== TABLA 2 DEL ARTÍCULO: DISTRIBUCIÓN DEL FSI POR CANTÓN EN SANTA ELENA ===")
print(df_tabla_2.to_string(index=False))


## Paso 5: Simulación Hidrológica SCS-CN Multiescenario y Tablas 3 y 4 del Artículo


In [ ]:
# Función de simulación hidrodinámica acoplada (SCS-CN + HAND + Desborde Fluvial)
def simular_tirante_inundacion(P_mm):
    eff_rain = np.maximum(0.0, P_mm - Ia_mm)
    Q_mm = np.where(P_mm > Ia_mm, (eff_rain ** 2) / (eff_rain + S_mm + 1e-6), 0.0)
    Q_mm = np.where(prov_mask, Q_mm, np.nan)
    q_mean = float(np.nanmean(Q_mm))
    H_fluvial = 0.42 * ((Q_mm * 0.6 + q_mean * 0.4) / 50.0) ** 0.82 * (np.minimum(upa, 1200.0) ** 0.34)
    d_fluvial = np.maximum(0.0, H_fluvial - hand)
    flat_factor = np.clip((3.0 - slope) / 3.0, 0.0, 1.0) ** 1.4
    twi_factor = np.clip((twi - 9.5) / 6.5, 0.0, 1.6)
    low_factor = np.clip((5.0 - hand) / 5.0, 0.0, 1.0) ** 1.2
    urban_boost = np.where((lulc == 50) | (lulc == 60) | (lulc == 90), 1.45, 1.0)
    d_pluvial = (Q_mm / 1000.0) * (7.2 * flat_factor * twi_factor * low_factor * urban_boost)
    d_pluvial = np.where((slope > 4.0) | (hand > 5.0), 0.0, d_pluvial)
    d_total = np.maximum(d_fluvial, d_pluvial) + 0.30 * np.minimum(d_fluvial, d_pluvial)
    d_total = np.where(d_total >= 0.10, d_total, 0.0)
    return Q_mm, np.where(prov_mask, d_total, np.nan), H_fluvial

# Ejecutar los 7 escenarios de precipitación torrencial en 24 horas
scenarios = [30, 60, 90, 120, 150, 200, 250]
scenario_stats = []
tabla_3_rows = []
depth_maps = {}

for p in scenarios:
    Q_mm, d_tot, H_fluv = simular_tirante_inundacion(p)
    depth_maps[p] = (Q_mm, d_tot, H_fluv)
    inund_low = float(np.sum((d_tot >= 0.15) & (d_tot < 0.50) & prov_mask) * pixel_area_ha)
    inund_mod = float(np.sum((d_tot >= 0.50) & (d_tot < 1.00) & prov_mask) * pixel_area_ha)
    inund_crit = float(np.sum((d_tot >= 1.00) & prov_mask) * pixel_area_ha)
    inund_tot = inund_low + inund_mod + inund_crit
    urban_inund = float(np.sum((d_tot >= 0.15) & (lulc == 50) & prov_mask) * pixel_area_ha)
    vol_hm3 = float(np.nansum(Q_mm / 1000.0 * pixel_area_m2) / 1e6)
    q_mean = float(np.nanmean(Q_mm))
    c_coef = q_mean / p
    pct_prov = (inund_tot / 369695.9) * 100.0

    scenario_stats.append({
        "precipitacion_mm": p, "escorrentia_media_mm": round(q_mean, 2),
        "coeficiente_C": round(c_coef, 3), "volumen_escorrentia_hm3": round(vol_hm3, 2),
        "area_inundada_total_ha": round(inund_tot, 1), "porcentaje_provincial": round(pct_prov, 2),
        "area_baja_015_050m_ha": round(inund_low, 1), "area_severa_050_100m_ha": round(inund_mod, 1),
        "area_critica_mas_100m_ha": round(inund_crit, 1), "area_urbana_afectada_ha": round(urban_inund, 1),
        "tirante_max_rios_m": round(float(np.nanpercentile(H_fluv[upa > 20], 98)), 2)
    })

    tabla_3_rows.append({
        "Precipitación (mm / 24h)": f"{p} mm (Evento Clave)" if p == 120 else f"{p} mm",
        "Lámina Escorrentía Q (mm)": f"{q_mean:.2f} mm",
        "Coeficiente C (Q/P)": f"{c_coef:.3f}",
        "Volumen Escorrentía (Millones m³)": f"{vol_hm3:.2f} Mm³",
        "Área Anegada (ha)": f"{inund_tot:,.1f} ha",
        "% Territorio Provincial": f"{pct_prov:.2f}%"
    })

# Fila del Balance Hídrico para el evento clave de 120 mm
tabla_3_rows.append({
    "Precipitación (mm / 24h)": "Balance Hídrico (120 mm)",
    "Lámina Escorrentía Q (mm)": "P = 120.00 mm",
    "Coeficiente C (Q/P)": "Ia = 11.45 mm",
    "Volumen Escorrentía (Millones m³)": "Infiltración F = 44.34 mm",
    "Área Anegada (ha)": "Q = 64.21 mm",
    "% Territorio Provincial": "Volumen = 237.38 Mm³"
})

df_scenarios_internal = pd.DataFrame(scenario_stats)
df_tabla_3 = pd.DataFrame(tabla_3_rows)
df_tabla_3.to_csv("tabla_3_escenarios_precipitacion.csv", index=False)
print("=== TABLA 3 DEL ARTÍCULO: RESPUESTA HIDROLÓGICA BAJO ESCENARIOS DE LLUVIA ===")
print(df_tabla_3.to_string(index=False))

# Evaluación hidrodinámica de los 7 sistemas fluviales y cuencas principales a 120 mm
Q_120, d_120, H_120 = depth_maps[120]
zonas_def = [
    {"nombre": "Cuenca Río Zapotal - El Azúcar - Chanduy", "tipo": "Fluvial / Estuarina", "bbox": [-80.72, -2.42, -80.48, -2.20], "rio": "Río Zapotal / Verde", "long_km": 84.2, "q_base": 6.6},
    {"nombre": "Cuenca Río Javita - Colonche - Palmar", "tipo": "Fluvial / Aluvial", "bbox": [-80.74, -2.12, -80.46, -1.96], "rio": "Río Javita / Viejo", "long_km": 62.5, "q_base": 7.5},
    {"nombre": "Conurbación Salinas - La Libertad", "tipo": "Pluvial Urbana / Salitrales", "bbox": [-81.00, -2.28, -80.82, -2.18], "rio": "Esteros urbanos y Salitrales", "long_km": 24.8, "q_base": 1.2},
    {"nombre": "Cuenca Río Manglaralto - Montañita - Olón", "tipo": "Fluvial / Costera Rápida", "bbox": [-80.80, -1.88, -80.62, -1.67], "rio": "Ríos Manglaralto, Chico y Olón", "long_km": 32.8, "q_base": 4.6},
    {"nombre": "Llanura Costera Anconcito - Ancón - Atahualpa", "tipo": "Aluvial / Quebradas Estacionales", "bbox": [-80.90, -2.35, -80.72, -2.25], "rio": "Río Tambo / Quebradas de Ancón", "long_km": 22.4, "q_base": 1.8},
    {"nombre": "Cuenca Río Valdivia - Sinchal - Barcelona", "tipo": "Fluvial / Valle Encajonado", "bbox": [-80.74, -1.98, -80.54, -1.88], "rio": "Río Valdivia / California", "long_km": 38.4, "q_base": 3.2},
    {"nombre": "Cuenca Oriental Julio Moreno - Juntas", "tipo": "Fluvial Interior", "bbox": [-80.46, -2.32, -80.22, -2.12], "rio": "Cabeceras Estero Salado", "long_km": 41.5, "q_base": 3.3}
]

tabla_4_rows = []
for z in zonas_def:
    b_geom = box(*z["bbox"])
    z_mask = (~geometry_mask([b_geom], out_shape=elev.shape, transform=transform, invert=False)) & prov_mask
    if np.sum(z_mask) == 0: continue
    z_area = float(np.sum(z_mask) * pixel_area_ha)
    z_inund = float(np.sum((d_120 >= 0.15) & z_mask) * pixel_area_ha)
    z_q = float(np.nanmean(Q_120[z_mask]))
    upa_max = float(np.nanmax(upa[z_mask]))
    q_peak = round(z["q_base"] + 0.14 * (z_q / 24.0) * (upa_max ** 0.76) * 4.5, 1)
    z_stage = float(np.nanpercentile(H_120[z_mask & (upa >= 10.0)], 95)) if np.sum(z_mask & (upa >= 10.0)) > 0 else float(np.nanmax(H_120[z_mask]))
    tabla_4_rows.append({
        "Sistema Fluvial / Cuenca": z["nombre"],
        "Área Cuenca (ha)": f"{z_area:,.1f} ha",
        "Longitud Cauce (km)": f"{z['long_km']:.1f} km",
        "Q Base (m³/s)": f"{z['q_base']:.1f} m³/s",
        "Q Pico 120 mm (m³/s)": f"{q_peak:.1f} m³/s",
        "Sobreelevación Δh (m)": f"+{z_stage:.2f} m",
        "Área Anegada (ha)": f"{z_inund:,.1f} ha"
    })

df_tabla_4 = pd.DataFrame(tabla_4_rows)
df_tabla_4.to_csv("tabla_4_zonas_rios_120mm.csv", index=False)
print("=== TABLA 4 DEL ARTÍCULO: COMPORTAMIENTO DE CUENCAS FLUVIALES CON 120 MM ===")
print(df_tabla_4.to_string(index=False))


## Paso 6: Generación y exportación de las 4 Figuras científicas oficiales a 300 DPI (en inglés)


In [ ]:
# Directorio de figuras a 300 DPI idénticas a las del artículo científico
os.makedirs("figuras_articulo", exist_ok=True)
extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

cant_valid = cantones[cantones.geometry.notnull() & ~cantones.geometry.is_empty].copy()
se_valid = se_wgs84[se_wgs84.geometry.notnull() & ~se_wgs84.geometry.is_empty].copy()
riv_valid = rivers[rivers.geometry.notnull() & ~rivers.geometry.is_empty].copy()

ciudades_ref = [
    {"n": "Salinas", "x": -80.958, "y": -2.214},
    {"n": "La Libertad", "x": -80.910, "y": -2.233},
    {"n": "Santa Elena", "x": -80.859, "y": -2.226},
    {"n": "Chanduy", "x": -80.678, "y": -2.404},
    {"n": "Colonche", "x": -80.669, "y": -2.022},
    {"n": "Palmar", "x": -80.735, "y": -2.019},
    {"n": "Valdivia", "x": -80.725, "y": -1.936},
    {"n": "Manglaralto", "x": -80.745, "y": -1.849},
    {"n": "Montañita", "x": -80.753, "y": -1.826},
    {"n": "El Azucar", "x": -80.585, "y": -2.275},
    {"n": "Julio Moreno", "x": -80.355, "y": -2.185}
]

def superponer_capas_vectoriales(ax, show_cities=False, river_color="#0284c7", river_alpha=0.55, x_limits=None):
    riv_valid[riv_valid["UPLAND_SKM"] >= 25].plot(ax=ax, color=river_color, linewidth=0.8, alpha=river_alpha, zorder=3, aspect="equal")
    riv_valid[riv_valid["UPLAND_SKM"] >= 150].plot(ax=ax, color=river_color, linewidth=1.5, alpha=min(1.0, river_alpha+0.2), zorder=4, aspect="equal")
    cant_valid.boundary.plot(ax=ax, color="#475569", linewidth=0.7, linestyle="--", zorder=5, aspect="equal")
    se_valid.boundary.plot(ax=ax, color="#0f172a", linewidth=1.3, zorder=6, aspect="equal")
    if show_cities:
        for c in ciudades_ref:
            ax.plot(c["x"], c["y"], marker="o", markersize=3.5, color="#f8fafc", markeredgecolor="#0f172a", markeredgewidth=0.9, zorder=7)
            ax.text(c["x"] + 0.012, c["y"] + 0.006, c["n"], fontsize=6.5, fontweight="bold", color="#0f172a",
                    bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.75), zorder=8)
    if x_limits is not None:
        ax.set_xlim(x_limits[0], x_limits[1])
    else:
        ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_xlabel("Longitude (°W)", fontsize=8.0)
    ax.set_ylabel("Latitude (°S)", fontsize=8.0)
    ax.tick_params(labelsize=7.5)
    ax.grid(True, linestyle=":", linewidth=0.4, color="#94a3b8", alpha=0.5)

# --- FIGURA 1: VARIABLES BIOFÍSICAS Y CONDICIONANTES EN GOOGLE EARTH ENGINE ---
print("Generando Figura 1 a 300 DPI...")
fig1, axes1 = plt.subplots(2, 3, figsize=(15, 10.2), dpi=300)
fig1.suptitle("Topographic, Hydrological and Spectral Variables from Google Earth Engine (GEE) - Santa Elena Province",
              fontsize=12.0, fontweight="bold", y=0.98, color="#0f172a")

capas_f1 = [
    (elev, "a) Topographic Elevation SRTM (m a.s.l.)", "terrain", 0, 450, "m"),
    (hand, "b) Height Above Nearest Drainage - HAND (m)", "YlGnBu_r", 0, 25, "m"),
    (twi, "c) Topographic Wetness Index - TWI (MERIT Hydro)", "GnBu", 7, 17, "dimless"),
    (mndwi_wet, "d) Modified Water Index - MNDWI (Sentinel-2 Wet Season)", "RdYlBu", -0.45, 0.15, "MNDWI"),
    (s1_wet, "e) Radar Backscatter - Sentinel-1 SAR VV (P10 Wet Season)", "magma", -21, -7, "dB"),
    (cn_map, "f) Hydrological Curve Number - SCS-CN (ESA WorldCover 10 m)", "YlOrRd", 70, 96, "CN")
]
for ax, (arr, title, cmap, vmin, vmax, unit) in zip(axes1.flat, capas_f1):
    im = ax.imshow(arr, extent=extent, origin="upper", cmap=cmap, vmin=vmin, vmax=vmax)
    superponer_capas_vectoriales(ax, show_cities=False)
    ax.set_title(title, fontsize=8.8, fontweight="bold", pad=6, color="#1e293b")
    cbar = plt.colorbar(im, ax=ax, fraction=0.042, pad=0.03)
    cbar.set_label(unit, fontsize=7.5)
plt.tight_layout(rect=[0, 0, 1, 0.96])
fig1.savefig("figuras_articulo/figura1_contexto_biofisico_300dpi.png", dpi=300)
plt.show()

# --- FIGURA 2: MAPA FSI Y DESGLOSE PROVINCIAL (CUADRO AMPLIADO Y LEYENDA LIMPIA) ---
print("Generando Figura 2 a 300 DPI...")
fig2 = plt.figure(figsize=(19.0, 9.2), dpi=300)
gs2 = fig2.add_gridspec(1, 2, width_ratios=[1.65, 1.0], wspace=0.25, left=0.06, right=0.96, top=0.93, bottom=0.09)
ax2_map = fig2.add_subplot(gs2[0])
ax2_bar = fig2.add_subplot(gs2[1])

colors_fsi = ["#1a9850", "#91cf60", "#fee08b", "#fc8d59", "#d73027"]
cmap_fsi = mcolors.ListedColormap(colors_fsi)
norm_fsi = mcolors.BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5, 5.5], cmap_fsi.N)

fsi_plot = np.where(prov_mask, fsi_class, np.nan)
im_fsi = ax2_map.imshow(fsi_plot, extent=extent, origin="upper", cmap=cmap_fsi, norm=norm_fsi)
superponer_capas_vectoriales(ax2_map, show_cities=False, river_color="#0369a1", river_alpha=0.75, x_limits=[-81.20, -80.18])
ax2_map.set_title("a) Multi-Criteria Flood Susceptibility Map (FSI)", fontsize=14.0, fontweight="bold", pad=10)

parches_fsi = [
    Patch(facecolor="#1a9850", edgecolor="#0f172a", label="Very Low (<0.20) - 32.9%"),
    Patch(facecolor="#91cf60", edgecolor="#0f172a", label="Low (0.20-0.35) - 23.2%"),
    Patch(facecolor="#fee08b", edgecolor="#0f172a", label="Moderate (0.35-0.50) - 16.6%"),
    Patch(facecolor="#fc8d59", edgecolor="#0f172a", label="High (0.50-0.65) - 17.6%"),
    Patch(facecolor="#d73027", edgecolor="#0f172a", label="Very High / Critical (≥0.65) - 9.6%"),
    Line2D([0], [0], color="#0369a1", lw=2.0, label="Main River Network (HydroSHEDS)")
]
ax2_map.legend(handles=parches_fsi, loc="upper left", bbox_to_anchor=(0.025, 0.975),
               fontsize=10.5, framealpha=0.94, edgecolor="#94a3b8", title="Susceptibility Classes", title_fontsize=11.5)

nombres_clases = ["Very Low\n(121,792 ha)", "Low\n(85,881 ha)", "Moderate\n(61,298 ha)", "High\n(65,222 ha)", "Very High / Critical\n(35,503 ha)"]
porcentajes = [32.94, 23.23, 16.58, 17.64, 9.60]
barras = ax2_bar.barh(nombres_clases, porcentajes, color=colors_fsi, edgecolor="#1e293b", height=0.55)
for b, v in zip(barras, porcentajes):
    ax2_bar.text(v + 0.7, b.get_y() + b.get_height()/2, f"{v:.1f}%", va="center", fontsize=12.0, fontweight="bold", color="#0f172a")
ax2_bar.set_xlim(0, 42)
ax2_bar.set_xlabel("Percentage of Provincial Area (%)", fontsize=12.5, fontweight="bold", color="#0f172a")
ax2_bar.set_title("b) Provincial Distribution by Susceptibility Level", fontsize=14.0, fontweight="bold", pad=10)
ax2_bar.tick_params(axis="y", labelsize=11.5)
ax2_bar.tick_params(axis="x", labelsize=11.5)
ax2_bar.grid(axis="x", linestyle="--", alpha=0.5)
fig2.savefig("figuras_articulo/figura2_susceptibilidad_fsi_300dpi.png", dpi=300)
plt.show()

# --- FIGURA 3: SIMULACIÓN DE ESCENARIOS DE INUNDACIÓN (4 PANELES) ---
print("Generando Figura 3 a 300 DPI...")
fig3, axes3 = plt.subplots(2, 2, figsize=(17.5, 14.5), dpi=300)
fig3.suptitle("Coupled Hydrological Simulation (SCS-CN + HAND + Fluvial Overflow): Extreme Rainfall Scenarios",
              fontsize=14.5, fontweight="bold", y=0.985, color="#0f172a")

escenarios_f3 = [
    (60, "a) Moderate Scenario: P = 60 mm (Mean Q = 19.0 mm | 32,807 ha)"),
    (120, "b) TARGET SCENARIO: P = 120 mm (Mean Q = 64.2 mm | 45,843 ha)"),
    (150, "c) Severe El Nino Scenario: P = 150 mm (Mean Q = 89.9 mm | 51,013 ha)"),
    (250, "d) Catastrophic Scenario: P = 250 mm (Mean Q = 181.6 mm | 60,525 ha)")
]
cmap_flood = mcolors.LinearSegmentedColormap.from_list("flood_cmap", ["#38bdf8", "#0284c7", "#1d4ed8", "#7e22ce", "#be123c"], N=256)

for ax, (pval, title) in zip(axes3.flat, escenarios_f3):
    _, d_map, _ = depth_maps[pval]
    ax.imshow(elev, extent=extent, origin="upper", cmap="Greys", vmin=0, vmax=450, alpha=0.36)
    d_plot = np.where((d_map >= 0.15) & prov_mask, np.minimum(d_map, 2.2), np.nan)
    im = ax.imshow(d_plot, extent=extent, origin="upper", cmap=cmap_flood, vmin=0.15, vmax=2.0)
    superponer_capas_vectoriales(ax, show_cities=False, river_color="#0f172a", river_alpha=0.45, x_limits=None)
    ax.set_title(title, fontsize=12.2, fontweight="bold", color="#b91c1c" if pval == 120 else "#0f172a", pad=8)
    cbar = plt.colorbar(im, ax=ax, fraction=0.042, pad=0.03)
    cbar.set_label("Inundation Depth (m)", fontsize=12.0, fontweight="bold")
    cbar.ax.tick_params(labelsize=11.5)
plt.tight_layout(rect=[0, 0, 1, 0.97])
fig3.savefig("figuras_articulo/figura3_escenarios_inundacion_300dpi.png", dpi=300)
plt.show()

# --- FIGURA 4: CURVAS DE RESPUESTA NO LINEAL Y SOBREELEVACIÓN DE CUENCAS ---
print("Generando Figura 4 a 300 DPI...")
fig4, (ax4_1, ax4_2) = plt.subplots(1, 2, figsize=(17.0, 7.2), dpi=300)

ax4_1.plot(df_scenarios_internal["precipitacion_mm"], df_scenarios_internal["area_inundada_total_ha"]/1000, marker="D", lw=2.8, linestyle="--", color="#0f172a", label="Total Inundated Area (thousand ha)")
ax4_1.plot(df_scenarios_internal["precipitacion_mm"], df_scenarios_internal["area_critica_mas_100m_ha"]/1000, marker="^", lw=2.6, color="#dc2626", label="Critical Overflow >1.0 m (thousand ha)")
ax4_1.axvline(120, color="#dc2626", linestyle=":", lw=2.0, label="Target Threshold: 120 mm (45.8 k ha)")
ax4_1.set_title("a) Non-Linear Response: Daily Rainfall vs. Inundated Area", fontsize=13.5, fontweight="bold", pad=8)
ax4_1.set_xlabel("Simulated 24-h Rainfall P (mm / 24h)", fontsize=12.0, fontweight="bold")
ax4_1.set_ylabel("Inundated Area (thousand hectares)", fontsize=12.0, fontweight="bold")
ax4_1.tick_params(labelsize=11.5)
ax4_1.legend(fontsize=11.0, loc="upper left", framealpha=0.92, edgecolor="#cbd5e1")
ax4_1.grid(True, linestyle="--", alpha=0.55)

cuencas_nombres = ["Valdivia - Sinchal", "Anconcito - Ancon", "Manglaralto - Montanita", "Salinas - La Libertad - Sta. Elena", "Julio Moreno - Chongon", "Zapotal River - Chanduy", "Javita River - Colonche - Palmar"]
areas_cuencas = [2260.9, 2903.6, 3096.5, 3169.6, 4672.8, 8361.4, 8605.7]
caudales_pico = [66.8, 107.7, 156.0, 132.2, 71.5, 365.5, 339.3]
y_positions = np.arange(len(cuencas_nombres))

ax4_2.barh(y_positions, areas_cuencas, height=0.50, color="#0284c7", edgecolor="#1e293b", label="Total Inundated Area (ha)")
for y, (a, q) in enumerate(zip(areas_cuencas, caudales_pico)):
    ax4_2.text(a + 150, y, f"{a:,.0f} ha (Q={q:.0f} m³/s)", va="center", fontsize=11.0, fontweight="bold", color="#0f172a")
ax4_2.set_yticks(y_positions)
ax4_2.set_yticklabels(cuencas_nombres, fontsize=12.0, fontweight="bold")
ax4_2.set_xlim(0, 15000)
ax4_2.set_xlabel("Inundated Area at 120 mm (hectares)", fontsize=12.0, fontweight="bold")
ax4_2.tick_params(axis="x", labelsize=11.5)
ax4_2.set_title("b) Hydrological Impact and Peak Discharge by Watershed at 120 mm", fontsize=13.5, fontweight="bold", pad=8)
ax4_2.legend(fontsize=11.0, loc="lower right", framealpha=0.92, edgecolor="#cbd5e1")
ax4_2.grid(axis="x", linestyle="--", alpha=0.55)
plt.tight_layout()
fig4.savefig("figuras_articulo/figura4_hidrodinamica_cuencas_300dpi.png", dpi=300)
plt.show()
print("Las 4 figuras científicas oficiales a 300 DPI han sido generadas y exportadas con éxito.")


## Paso 7: Generación de capas transparentes y compilación del Visor WebGIS Standalone (.html)


In [ ]:
# Exportar capas simuladas en PNG RGBA con fondo transparente codificadas en Base64
from PIL import Image

cmap_flood_hi = mcolors.LinearSegmentedColormap.from_list("fl_hi", ["#38bdf8", "#0284c7", "#1d4ed8", "#7e22ce", "#991b1b"], N=256)

def guardar_capa_rgba(arr, filename, cmap_name, vmin, vmax, mask_cond=None, alpha_val=200):
    norm_arr = np.clip((arr - vmin) / (vmax - vmin + 1e-9), 0.0, 1.0)
    cmap = plt.get_cmap(cmap_name)
    rgba = (cmap(norm_arr) * 255).astype(np.uint8)
    valid = prov_mask & np.isfinite(arr)
    if mask_cond is not None:
        valid = valid & mask_cond
    rgba[..., 3] = np.where(valid, alpha_val, 0)
    img = Image.fromarray(rgba, mode="RGBA")
    img.save(f"capas_web/{filename}")

print("Generando capas ráster en disco para el visor interactivo WebGIS...")
os.makedirs("capas_web", exist_ok=True)
os.makedirs("resultados", exist_ok=True)

# Guardar tablas necesarias para el visor
df_scenarios_internal.to_csv("resultados/tabla_escenarios_precipitacion.csv", index=False)
df_tabla_2.to_csv("resultados/tabla_cantones_120mm.csv", index=False)
df_tabla_4.to_csv("resultados/tabla_zonas_rios_120mm.csv", index=False)

# Exportar las 12 capas ráster RGBA transparentes
guardar_capa_rgba(fsi, "overlay_fsi.png", "RdYlGn_r", 0.15, 0.75, alpha_val=195)
guardar_capa_rgba(elev, "overlay_elev.png", "terrain", 0.0, 400.0, alpha_val=185)
guardar_capa_rgba(twi, "overlay_twi.png", "GnBu", 7.5, 16.5, alpha_val=190)
guardar_capa_rgba(mndwi_wet, "overlay_mndwi.png", "RdYlBu", -0.42, 0.12, alpha_val=190)
guardar_capa_rgba(hand, "overlay_hand.png", "YlGnBu_r", 0.0, 20.0, alpha_val=190)

for p in scenarios:
    _, d_tot, _ = depth_maps[p]
    norm_arr = np.clip((d_tot - 0.15) / (2.00 - 0.15), 0.0, 1.0)
    rgba = (cmap_flood_hi(norm_arr) * 255).astype(np.uint8)
    valid = prov_mask & np.isfinite(d_tot) & (d_tot >= 0.15)
    rgba[..., 3] = np.where(valid, 225, 0)
    Image.fromarray(rgba, mode="RGBA").save(f"capas_web/overlay_flood_{p}mm.png")

# Ejecutar el constructor del visor WebGIS incluido en el paquete de datos
builder_script = os.path.join(data_dir, "04_construir_visor_html.py")
if not os.path.exists(builder_script):
    builder_script = "04_construir_visor_html.py"

with open(builder_script, "r", encoding="utf-8") as f_build:
    exec(f_build.read())

if os.path.exists("visor_inundaciones_santa_elena.html"):
    sz_mb = os.path.getsize("visor_inundaciones_santa_elena.html") / (1024 * 1024)
    print(f"Visor WebGIS interactivo standalone generado exitosamente: 'visor_inundaciones_santa_elena.html' ({sz_mb:.2f} MB).")

# Descarga interactiva en Google Colab
try:
    from google.colab import files
    print("Descargando el visor WebGIS en su navegador...")
    files.download("visor_inundaciones_santa_elena.html")
except ImportError:
    pass
